In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
from vascx.faz.retina import FAZRetina

from rtnls_enface.utils.data_loading import open_mask
from vascx.faz.features.bifurcations import BifurcationCount
from vascx.faz.features.caliber import Caliber
from vascx.faz.features.tortuosity import Tortuosity
from vascx.faz.features.vascular_densities import VascularDensity
from rtnls_enface.utils.plotting import plot_columns, plot_rows
from rtnls_enface.grids.etdrs import Ring

In [ ]:
ds_path = Path('/mnt/oogergo/eyened/sinergia/faz_samples/FAZ')
images_paths = sorted(list(ds_path.glob('*_OCTA.png')))
masks_paths = sorted(list(ds_path.glob('*_AV_Map.png')))
faz_paths = sorted(list(ds_path.glob('*_FAZ.png')))

In [ ]:
def av_loader(fpath, *args):
    im = open_mask(fpath)

    if len(im.shape) == 3:
        im = im[...,0]

    return {
        "arteries": (im == 127),
        "veins": (im==255),
    }

def load_retina(index):
    return FAZRetina.from_file(av_path=masks_paths[index], faz_path=faz_paths[index], image_path=images_paths[index], av_loader=av_loader)

In [ ]:
ret = load_retina(1)

In [ ]:
from vascx.faz.features.faz_params import FazParameter, FazParameterType


feature = FazParameter(FazParameterType.PerimeterLength)

In [ ]:
fig, ax = plt.subplots()
feature.plot(ax, ret.arteries)

In [ ]:
def plot_row(axs, row, fig):
    ret = load_retina(row)
    VascularDensity(grid_field=Ring.Inner).plot(axs[0], ret.arteries)
    VascularDensity(grid_field=Ring.Inner).plot(axs[1], ret.veins)
    VascularDensity(grid_field=Ring.Outer).plot(axs[2], ret.arteries)
    VascularDensity(grid_field=Ring.Outer).plot(axs[3], ret.veins)

plot_rows(plot_row, 4, 4)

In [ ]:
def plot_column(axs, col, fig):
    ret = load_retina(col)
    BifurcationCount().plot(axs[0], ret.veins)
    Caliber(min_numpoints=3).plot(axs[1], ret.veins)
    Tortuosity(min_numpoints=5).plot(axs[2], ret.veins)
    VascularDensity().plot(axs[3], ret.veins)

plot_columns(plot_column, 4, 4)